# 🏃🚴 Strava Activiteiten Analyse

Analyse van GPS-activiteiten uit `.fit` en `.gpx` bestanden.

**Installeer eerst de benodigde packages (eenmalig):**
```
pip install fitparse gpxpy haversine folium pyproj numpy pandas matplotlib seaborn
```

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import folium
import gpxpy
from fitparse import FitFile
from haversine import haversine, Unit
from pyproj import Transformer
from datetime import timezone, timedelta

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('✅ Alle packages geladen')

---
## 📂 Stap 0 – Activiteiten inladen

We ondersteunen zowel `.fit` als `.gpx` bestanden.  
Pas het pad hieronder aan naar je eigen Strava-map.

In [ ]:
# ── Configuratie ────────────────────────────────────────────────────────────
# Pad naar de map met jouw .fit / .gpx bestanden:
DATA_DIR = 'data/'          # ← pas aan naar jouw map

# Of geef losse bestanden op:
LOSSE_BESTANDEN = [
    'data/Middagloop.fit',
    'data/10036442137.gpx',
]

# Tijdzone correctie (Strava slaat UTC op; Amsterdam = UTC+1 of UTC+2)
LOCAL_TZ = timezone(timedelta(hours=2))  # zomertijd (CEST)
# LOCAL_TZ = timezone(timedelta(hours=1))  # wintertijd (CET)

# GPS-snelheidsdrempel voor outlier-detectie (km/h)
MAX_SNELHEID_KMH = 120.0

In [ ]:
# ── Parser: .fit ────────────────────────────────────────────────────────────
def laad_fit(pad: str) -> pd.DataFrame:
    """Lees een .fit bestand in als DataFrame."""
    ff = FitFile(pad)
    rijen = []
    for record in ff.get_messages('record'):
        rij = {field.name: field.value for field in record}
        rijen.append(rij)
    df = pd.DataFrame(rijen)
    if df.empty:
        return df

    # Kolomnamen normaliseren
    hernoem = {
        'position_lat':  'lat_semicircles',
        'position_long': 'lon_semicircles',
        'enhanced_altitude': 'hoogte',
        'altitude': 'hoogte',
    }
    df = df.rename(columns={k: v for k, v in hernoem.items() if k in df.columns})

    # .fit slaat lat/lon op als semicircles → graden
    if 'lat_semicircles' in df.columns:
        df['lat'] = df['lat_semicircles'] * (180 / 2**31)
        df['lon'] = df['lon_semicircles'] * (180 / 2**31)

    # Tijdstempel
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

    # Probeer activiteitstype uit de sessie te lezen
    sport = 'onbekend'
    for sessie in ff.get_messages('session'):
        for veld in sessie:
            if veld.name == 'sport':
                sport = str(veld.value)
    df['sport'] = sport
    df['bron'] = os.path.basename(pad)
    return df


# ── Parser: .gpx ────────────────────────────────────────────────────────────
def laad_gpx(pad: str) -> pd.DataFrame:
    """Lees een .gpx bestand in als DataFrame."""
    with open(pad, 'r') as f:
        gpx = gpxpy.parse(f)
    rijen = []
    sport = 'onbekend'
    for track in gpx.tracks:
        if track.type:
            sport = track.type
        for segment in track.segments:
            for pt in segment.points:
                rijen.append({
                    'timestamp': pd.Timestamp(pt.time).tz_localize('UTC')
                                 if pt.time.tzinfo is None
                                 else pd.Timestamp(pt.time).tz_convert('UTC'),
                    'lat':    pt.latitude,
                    'lon':    pt.longitude,
                    'hoogte': pt.elevation,
                })
    df = pd.DataFrame(rijen)
    df['sport'] = sport
    df['bron']  = os.path.basename(pad)
    return df


# ── Alle bestanden inladen ──────────────────────────────────────────────────
def verzamel_bestanden():
    paden = list(LOSSE_BESTANDEN)
    if os.path.isdir(DATA_DIR):
        paden += glob.glob(os.path.join(DATA_DIR, '*.fit'))
        paden += glob.glob(os.path.join(DATA_DIR, '*.gpx'))
    return list(dict.fromkeys(paden))  # dedupliceer

activiteiten_raw = []
for pad in verzamel_bestanden():
    if not os.path.isfile(pad):
        print(f'⚠️  Bestand niet gevonden: {pad}')
        continue
    try:
        df = laad_fit(pad) if pad.endswith('.fit') else laad_gpx(pad)
        if not df.empty:
            activiteiten_raw.append(df)
            print(f'✅ {os.path.basename(pad):40s}  {len(df):5d} punten  sport={df["sport"].iloc[0]}')
    except Exception as e:
        print(f'❌ Fout bij {pad}: {e}')

print(f'\nTotaal: {len(activiteiten_raw)} activiteit(en) geladen')

---
## 📏 Opdracht 1 – Afstand & snelheid per datapunt

We berekenen de afstand tussen opeenvolgende punten op twee manieren:
- **Haversine** – houdt rekening met de bolrond vorm van de aarde
- **RD-coördinaten (EPSG:28992)** – stelling van Pythagoras in meters (voor Nederland nauwkeurig genoeg)

In [ ]:
# Transformer WGS84 → Rijksdriehoek
_wgs_naar_rd = Transformer.from_crs('EPSG:4326', 'EPSG:28992', always_xy=True)

def bereken_kinematics(df: pd.DataFrame) -> pd.DataFrame:
    """Voeg afstand, tijdsverschil en snelheidskolommen toe."""
    df = df.copy().sort_values('timestamp').reset_index(drop=True)

    # Alleen rijen met geldige GPS
    heeft_gps = df['lat'].notna() & df['lon'].notna()

    # ── Methode 1: Haversine ────────────────────────────────────────────────
    lat = df['lat'].values
    lon = df['lon'].values
    afstand_hav = np.full(len(df), np.nan)
    for i in range(1, len(df)):
        if heeft_gps.iloc[i] and heeft_gps.iloc[i - 1]:
            afstand_hav[i] = haversine(
                (lat[i - 1], lon[i - 1]),
                (lat[i],     lon[i]),
                unit=Unit.METERS
            )
    df['afstand_hav_m'] = afstand_hav

    # ── Methode 2: RD-coördinaten + Pythagoras ─────────────────────────────
    geldige_idx = df.index[heeft_gps].tolist()
    rd_x, rd_y = _wgs_naar_rd.transform(lon, lat)
    afstand_rd = np.full(len(df), np.nan)
    for i in range(1, len(df)):
        if heeft_gps.iloc[i] and heeft_gps.iloc[i - 1]:
            dx = rd_x[i] - rd_x[i - 1]
            dy = rd_y[i] - rd_y[i - 1]
            afstand_rd[i] = np.sqrt(dx**2 + dy**2)
    df['afstand_rd_m'] = afstand_rd

    # ── Tijdsverschil ──────────────────────────────────────────────────────
    df['tijd_s'] = df['timestamp'].diff().dt.total_seconds()

    # ── Snelheid ───────────────────────────────────────────────────────────
    df['snelheid_ms']  = df['afstand_hav_m'] / df['tijd_s']
    df['snelheid_kmh'] = df['snelheid_ms'] * 3.6

    return df


# Pas toe op alle activiteiten
activiteiten = [bereken_kinematics(df) for df in activiteiten_raw]

# Toon eerste activiteit als voorbeeld
if activiteiten:
    voorbeeld = activiteiten[0]
    print(f'Activiteit: {voorbeeld["bron"].iloc[0]}')
    print(f'Sport:      {voorbeeld["sport"].iloc[0]}')
    cols = ['timestamp', 'lat', 'lon', 'afstand_hav_m', 'afstand_rd_m', 'tijd_s', 'snelheid_kmh']
    aanwezig = [c for c in cols if c in voorbeeld.columns]
    display(voorbeeld[aanwezig].head(10))

    # Vergelijk de twee afstandsmethoden
    verschil = (voorbeeld['afstand_hav_m'] - voorbeeld['afstand_rd_m']).abs().mean()
    print(f'\nGemiddeld verschil Haversine vs RD: {verschil:.4f} m (verwacht < 1 cm voor NL)')

---
## 🔍 Opdracht 2 – Foutcontrole & outlier-verwijdering

We controleren op:
1. **Onrealistische snelheden** (GPS-sprongen)
2. **Dubbele of ontbrekende tijdstempels**
3. **Tijdzone** – Strava slaat UTC op; we zetten om naar lokale tijd

In [ ]:
def controleer_en_reinig(df: pd.DataFrame, max_snelheid: float = MAX_SNELHEID_KMH) -> pd.DataFrame:
    """Detecteer en verwijder verdachte GPS-punten."""
    n_voor = len(df)
    bron = df['bron'].iloc[0]

    # ── 1. Tijdzone-check ──────────────────────────────────────────────────
    if df['timestamp'].dt.tz is None:
        print(f'⚠️  {bron}: tijdstempels hebben geen tijdzone → UTC aangenomen')
        df['timestamp'] = df['timestamp'].dt.tz_localize('UTC')

    # Kolom met lokale tijd
    df['timestamp_lokaal'] = df['timestamp'].dt.tz_convert(LOCAL_TZ)
    print(f'🕐 {bron}: eerste punt UTC={df["timestamp"].iloc[0]}  '
          f'lokaal={df["timestamp_lokaal"].iloc[0]}')

    # ── 2. Dubbele tijdstempels ────────────────────────────────────────────
    n_dubbel = df.duplicated('timestamp').sum()
    if n_dubbel:
        print(f'⚠️  {bron}: {n_dubbel} dubbele tijdstempels → verwijderd')
        df = df.drop_duplicates('timestamp')

    # ── 3. Ontbrekende GPS ─────────────────────────────────────────────────
    n_mis_gps = df['lat'].isna().sum()
    if n_mis_gps:
        print(f'⚠️  {bron}: {n_mis_gps} punten zonder GPS-coördinaten')

    # ── 4. Snelheidsoutliers ───────────────────────────────────────────────
    outlier_mask = df['snelheid_kmh'] > max_snelheid
    n_outlier = outlier_mask.sum()
    if n_outlier:
        print(f'⚠️  {bron}: {n_outlier} punten met snelheid > {max_snelheid} km/h '
              f'(max gevonden: {df["snelheid_kmh"].max():.1f} km/h) → verwijderd')
        df = df[~outlier_mask]

    # ── 5. Negatief tijdsverschil ──────────────────────────────────────────
    df = df.sort_values('timestamp').reset_index(drop=True)
    n_achteruit = (df['tijd_s'] < 0).sum()
    if n_achteruit:
        print(f'⚠️  {bron}: {n_achteruit} punten met negatief tijdsverschil → verwijderd')
        df = df[df['tijd_s'] >= 0]

    n_na = len(df)
    print(f'✅ {bron}: {n_voor} → {n_na} punten ({n_voor - n_na} verwijderd)\n')
    return df.reset_index(drop=True)


activiteiten_schoon = [controleer_en_reinig(df) for df in activiteiten]

# Visualiseer snelheidsverdeling vóór/na reiniging (eerste activiteit)
if activiteiten and activiteiten_schoon:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, data, titel in zip(
        axes,
        [activiteiten[0], activiteiten_schoon[0]],
        ['Vóór reiniging', 'Na reiniging']
    ):
        v = data['snelheid_kmh'].dropna()
        ax.hist(v, bins=60, color='steelblue', edgecolor='white')
        ax.axvline(MAX_SNELHEID_KMH, color='red', linestyle='--', label=f'Drempel {MAX_SNELHEID_KMH} km/h')
        ax.set_xlabel('Snelheid (km/h)')
        ax.set_ylabel('Aantal punten')
        ax.set_title(f'{titel} – {data["bron"].iloc[0]}')
        ax.legend()
    plt.tight_layout()
    plt.show()

---
## 📊 Opdracht 3 – Totale afstand, duur & gemiddelde snelheid per activiteit

In [ ]:
def activiteit_samenvatting(df: pd.DataFrame) -> dict:
    """Bereken samenvattende statistieken voor één activiteit."""
    totaal_afstand_km = df['afstand_hav_m'].sum() / 1000
    start = df['timestamp'].min()
    eind  = df['timestamp'].max()
    duur_min = (eind - start).total_seconds() / 60
    gem_snelheid = totaal_afstand_km / (duur_min / 60) if duur_min > 0 else np.nan
    max_snelheid = df['snelheid_kmh'].max()

    hoogteverschil = np.nan
    if 'hoogte' in df.columns and df['hoogte'].notna().any():
        positief = df['hoogte'].diff().clip(lower=0).sum()
        hoogteverschil = positief

    return {
        'bestand':           df['bron'].iloc[0],
        'sport':             df['sport'].iloc[0],
        'datum':             start.strftime('%Y-%m-%d'),
        'starttijd':         start.strftime('%H:%M:%S'),
        'afstand_km':        round(totaal_afstand_km, 2),
        'duur_min':          round(duur_min, 1),
        'gem_snelheid_kmh':  round(gem_snelheid, 2),
        'max_snelheid_kmh':  round(max_snelheid, 1),
        'hoogteklimmen_m':   round(hoogteverschil, 0) if not np.isnan(hoogteverschil) else np.nan,
        'n_punten':          len(df),
    }


samenvattingen = [activiteit_samenvatting(df) for df in activiteiten_schoon]
df_samen = pd.DataFrame(samenvattingen)

# Mooie weergave
display(df_samen.style
    .format({
        'afstand_km':       '{:.2f} km',
        'duur_min':         '{:.0f} min',
        'gem_snelheid_kmh': '{:.1f} km/h',
        'max_snelheid_kmh': '{:.1f} km/h',
        'hoogteklimmen_m':  '{:.0f} m',
    })
    .background_gradient(subset=['afstand_km', 'gem_snelheid_kmh'], cmap='Blues')
)

---
## 🗺️ Opdracht 4 – Kaartje met route & rijrichting

We plotten de route op een interactieve kaart met pijlen die de rijrichting aangeven.  
Het kaartje wordt opgeslagen als HTML-bestand.

In [ ]:
def maak_kaart(df: pd.DataFrame, uitvoerpad: str = None) -> folium.Map:
    """Maak een interactieve Folium-kaart met de route en richting."""
    df_gps = df.dropna(subset=['lat', 'lon']).copy()
    if df_gps.empty:
        print('Geen GPS-data beschikbaar voor kaart.')
        return None

    # Centreerpunt
    midden_lat = df_gps['lat'].mean()
    midden_lon = df_gps['lon'].mean()

    kaart = folium.Map(
        location=[midden_lat, midden_lon],
        zoom_start=15,
        tiles='CartoDB positron'
    )

    # ── Kleurcodering op snelheid ──────────────────────────────────────────
    v = df_gps['snelheid_kmh'].fillna(0).clip(lower=0)
    v_max = v.quantile(0.95) if v.max() > 0 else 1
    norm  = mcolors.Normalize(vmin=0, vmax=v_max)
    cmap  = cm.get_cmap('RdYlGn')

    coördinaten = list(zip(df_gps['lat'], df_gps['lon']))

    # Segmenten met kleur op snelheid
    for i in range(1, len(df_gps)):
        snelheid = v.iloc[i]
        kleur = mcolors.to_hex(cmap(norm(snelheid)))
        folium.PolyLine(
            [coördinaten[i - 1], coördinaten[i]],
            color=kleur, weight=4, opacity=0.85,
            tooltip=f'{snelheid:.1f} km/h'
        ).add_to(kaart)

    # ── Richtingspijlen (elke 30 punten) ──────────────────────────────────
    stap = max(1, len(df_gps) // 40)
    for i in range(stap, len(df_gps), stap):
        lat1, lon1 = coördinaten[i - 1]
        lat2, lon2 = coördinaten[i]
        folium.RegularPolygonMarker(
            location=[(lat1 + lat2) / 2, (lon1 + lon2) / 2],
            number_of_sides=3,
            radius=6,
            rotation=np.degrees(np.arctan2(lon2 - lon1, lat2 - lat1)),
            color='navy', fill=True, fill_color='navy', fill_opacity=0.7,
            weight=1
        ).add_to(kaart)

    # ── Start- en eindmarker ───────────────────────────────────────────────
    folium.Marker(
        coördinaten[0],
        popup='<b>Start</b>',
        icon=folium.Icon(color='green', icon='play', prefix='fa')
    ).add_to(kaart)
    folium.Marker(
        coördinaten[-1],
        popup='<b>Finish</b>',
        icon=folium.Icon(color='red', icon='flag', prefix='fa')
    ).add_to(kaart)

    # Legenda
    legenda_html = f'''
    <div style="position:fixed;bottom:40px;left:40px;background:white;
         padding:10px 14px;border-radius:8px;box-shadow:2px 2px 6px rgba(0,0,0,.3);
         font-family:Arial;font-size:13px;z-index:9999">
      <b>{df_gps["bron"].iloc[0]}</b><br>
      <span style="color:red">■</span> Langzaam &nbsp;
      <span style="color:orange">■</span> Gemiddeld &nbsp;
      <span style="color:green">■</span> Snel<br>
      Max snelheid (p95): {v_max:.1f} km/h
    </div>'''
    kaart.get_root().html.add_child(folium.Element(legenda_html))

    if uitvoerpad:
        kaart.save(uitvoerpad)
        print(f'✅ Kaart opgeslagen: {uitvoerpad}')

    return kaart


# Maak kaart voor elke activiteit
for i, df in enumerate(activiteiten_schoon):
    bron = df['bron'].iloc[0].replace('.fit', '').replace('.gpx', '')
    kaart = maak_kaart(df, uitvoerpad=f'kaart_{bron}.html')

# Toon laatste kaart inline in het notebook
if activiteiten_schoon:
    kaart = maak_kaart(activiteiten_schoon[-1])
    display(kaart)

---
## 📈 Opdracht 5 – Histogram gemiddelde snelheid & activiteitsgroepen

Typische snelheden per sporttype:
| Sport | Gemiddelde snelheid |
|---|---|
| Zwemmen | 1–4 km/h |
| Wandelen | 4–6 km/h |
| Hardlopen | 6–15 km/h |
| Wielrennen | 15–45 km/h |
| Overig | > 45 km/h |

In [ ]:
# ── Groepering op activiteitstype (op basis van snelheid + metadata) ────────
SNELHEID_GRENZEN = [
    (0,  4,   'zwemmen',    '#4FC3F7'),
    (4,  6,   'wandelen',   '#81C784'),
    (6,  15,  'hardlopen',  '#FFB74D'),
    (15, 45,  'wielrennen', '#E57373'),
    (45, 999, 'overig/auto','#CE93D8'),
]

def bepaal_groep(row):
    """Classificeer op basis van gemiddelde snelheid."""
    # Geef voorrang aan sporttype uit het bestand
    sport_raw = str(row.get('sport', '')).lower()
    mapping = {
        'running': 'hardlopen', 'run': 'hardlopen',
        'cycling': 'wielrennen', 'ride': 'wielrennen', 'biking': 'wielrennen',
        'swimming': 'zwemmen', 'swim': 'zwemmen',
        'walking': 'wandelen', 'hike': 'wandelen', 'hiking': 'wandelen',
    }
    for sleutel, groep in mapping.items():
        if sleutel in sport_raw:
            return groep

    # Terugval: snelheidsindeling
    v = row.get('gem_snelheid_kmh', np.nan)
    if pd.isna(v):
        return 'onbekend'
    for v_min, v_max, groep, _ in SNELHEID_GRENZEN:
        if v_min <= v < v_max:
            return groep
    return 'onbekend'


df_samen['groep'] = df_samen.apply(bepaal_groep, axis=1)

print('Activiteiten per groep:')
display(df_samen[['bestand', 'sport', 'gem_snelheid_kmh', 'groep']])


# ── Plot 1: Histogram punt-snelheden voor alle activiteiten ─────────────────
KLEUR_MAP = {
    'zwemmen':    '#4FC3F7',
    'wandelen':   '#81C784',
    'hardlopen':  '#FFB74D',
    'wielrennen': '#E57373',
    'overig/auto':'#CE93D8',
    'onbekend':   '#BDBDBD',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Links: per activiteit, gekleurd per groep
ax = axes[0]
for df, info in zip(activiteiten_schoon, samenvattingen):
    groep = df_samen.loc[df_samen['bestand'] == info['bestand'], 'groep'].values[0]
    kleur = KLEUR_MAP.get(groep, '#BDBDBD')
    v = df['snelheid_kmh'].dropna()
    v = v[v < MAX_SNELHEID_KMH]
    ax.hist(v, bins=50, alpha=0.7, color=kleur, edgecolor='white',
            label=f'{info["bestand"]} ({groep})')
ax.set_xlabel('Snelheid (km/h)')
ax.set_ylabel('Aantal GPS-punten')
ax.set_title('Snelheidsverdeling per activiteit')
ax.legend(fontsize=8)

# Voeg grenzen toe als verticale lijnen
for v_min, v_max, groep, kleur in SNELHEID_GRENZEN:
    ax.axvline(v_min, color='gray', linestyle=':', alpha=0.5)

# Rechts: gemiddelde snelheid per groep (barplot)
ax2 = axes[1]
groep_stats = df_samen.groupby('groep')['gem_snelheid_kmh'].mean().reset_index()
kleuren = [KLEUR_MAP.get(g, '#BDBDBD') for g in groep_stats['groep']]
bars = ax2.bar(groep_stats['groep'], groep_stats['gem_snelheid_kmh'],
               color=kleuren, edgecolor='white', linewidth=1.5)
ax2.bar_label(bars, fmt='%.1f km/h', padding=3, fontsize=9)
ax2.set_ylabel('Gemiddelde snelheid (km/h)')
ax2.set_xlabel('Activiteitstype')
ax2.set_title('Gemiddelde snelheid per sportgroep')

plt.suptitle('Activiteitsanalyse – Snelheidsgroepen', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('snelheid_histogram.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Histogram opgeslagen als snelheid_histogram.png')

In [ ]:
# ── Bonus: snelheid over tijd per activiteit ────────────────────────────────
if activiteiten_schoon:
    n = len(activiteiten_schoon)
    fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), squeeze=False)

    for i, (df, info) in enumerate(zip(activiteiten_schoon, samenvattingen)):
        ax = axes[i][0]
        groep = df_samen.loc[df_samen['bestand'] == info['bestand'], 'groep'].values[0]
        kleur = KLEUR_MAP.get(groep, 'steelblue')

        # Gesmoothde snelheid (voortschrijdend gemiddelde)
        v_smooth = df['snelheid_kmh'].rolling(window=15, center=True, min_periods=1).mean()
        t = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds() / 60  # minuten

        ax.fill_between(t, v_smooth, alpha=0.25, color=kleur)
        ax.plot(t, v_smooth, color=kleur, linewidth=1.5)
        ax.axhline(info['gem_snelheid_kmh'], color='gray', linestyle='--',
                   label=f'Gemiddeld: {info["gem_snelheid_kmh"]:.1f} km/h')
        ax.set_ylabel('Snelheid (km/h)')
        ax.set_xlabel('Tijd (min)')
        ax.set_title(f'{info["bestand"]}  |  {groep}  |  {info["afstand_km"]} km  |  {info["duur_min"]:.0f} min')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('snelheid_over_tijd.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('✅ Tijdgrafiek opgeslagen als snelheid_over_tijd.png')

---
## 📋 Samenvatting

| Opdracht | Wat er is gedaan |
|---|---|
| 1 | Haversine-afstand én RD-Pythagoras berekend per GPS-punt |
| 2 | Outliers verwijderd op snelheid, dubbele tijdstempels, UTC→lokale tijd |
| 3 | Totale afstand, duur, gem./max snelheid en hoogteklimmen per activiteit |
| 4 | Interactieve Folium-kaart met kleurcodering op snelheid en richtingspijlen |
| 5 | Histogram + classificatie in zwemmen / wandelen / hardlopen / wielrennen |